# SecureSpeak — Phase 2, Item 2.1: PhishTank External Validation Setup

## Purpose
Download fresh real-world phishing URLs from PhishTank and prepare them as an **external test set** for your URL classifier. This kills the "the authors only tested on data they collected themselves" critique.

## Why This Matters
Right now your 99.76% URL accuracy is on StealthPhisher 2025 — the same distribution your model trained on. A Q1 reviewer will write: *"all URL evaluations are on a single corpus collected at training time."* That's a reject signal.

After this notebook runs, you will have:
- A **fresh PhishTank phishing URL set** your model has never seen
- A **clean benign URL set** (Tranco Top sites) for negative class
- A combined test CSV ready for scoring in Item 2.2

**Expected honest result:** Your accuracy will drop from 99.76% to somewhere between 75% and 92% on PhishTank. **This drop is what reviewers respect.** A 99% number on an external test set is suspicious; an honest 85% with explained limitations is publishable.

## Folder Layout
```
cse498R/
├── Datasets/                              ← read-only inputs
└── model_for_research/
    └── phase2_external/                   ← this notebook writes here
        ├── phishtank_raw.csv              ← fresh download from PhishTank
        ├── tranco_benign_raw.csv          ← legitimate URLs from Tranco
        ├── external_url_testset.csv       ← combined cleaned test set
        ├── reports/
        │   └── phishtank_acquisition.txt  ← provenance & stats
        └── phase2_1_summary.json          ← machine-readable summary
```

## What This Notebook Does NOT Do
- Does not train anything
- Does not score URLs with your model (that's Item 2.2)
- Does not modify any existing dataset

---
## Step 0 — Environment Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, json, io, gzip, urllib.request, urllib.error, ssl, zipfile
from pathlib import Path
from datetime import datetime
from urllib.parse import urlparse

import numpy as np
import pandas as pd

# ── PATHS ───────────────────────────────────────────────────────────────────
BASE_DATA      = '/content/drive/MyDrive/cse498R/Datasets'
BASE_OUT       = '/content/drive/MyDrive/cse498R/model_for_research'
PHASE2_ROOT    = os.path.join(BASE_OUT, 'phase2_external')
REPORTS_DIR    = os.path.join(PHASE2_ROOT, 'reports')

PHISHTANK_RAW  = os.path.join(PHASE2_ROOT, 'phishtank_raw.csv')
TRANCO_RAW     = os.path.join(PHASE2_ROOT, 'tranco_benign_raw.csv')
EXTERNAL_TEST  = os.path.join(PHASE2_ROOT, 'external_url_testset.csv')
PROVENANCE_TXT = os.path.join(REPORTS_DIR, 'phishtank_acquisition.txt')
SUMMARY_JSON   = os.path.join(PHASE2_ROOT, 'phase2_1_summary.json')

for d in [PHASE2_ROOT, REPORTS_DIR]:
    Path(d).mkdir(parents=True, exist_ok=True)

print('═' * 70)
print(' PHASE 2.1 — PHISHTANK EXTERNAL VALIDATION SETUP')
print('═' * 70)
print(f'Input  (Datasets):           {BASE_DATA}')
print(f'Output (phase2_external):    {PHASE2_ROOT}')
print(f'Today: {datetime.now().strftime("%Y-%m-%d %H:%M")}')

---
## Step 1 — Download PhishTank Online Valid Database

**Source:** `http://data.phishtank.com/data/online-valid.csv` — community-verified phishing URLs, updated hourly.

**Why this URL specifically:** Only verified phishing URLs that are currently online. We do not want unverified submissions (too noisy) or offline URLs (already taken down, not representative of live threats).

**Note on rate-limiting:** PhishTank serves this file freely without API key but rate-limits aggressive requests. If you get a 403 or 429 error, just wait 15 minutes and re-run.

In [ ]:
PHISHTANK_URL = 'http://data.phishtank.com/data/online-valid.csv'

# PhishTank requires a User-Agent string per their terms of service
headers = {
    'User-Agent': 'phishtank/SecureSpeak-NSU-Research-2026'
}

print(f'Downloading from: {PHISHTANK_URL}')
print('(this is a community feed updated hourly — file is usually 5-15 MB)')
print()

try:
    req = urllib.request.Request(PHISHTANK_URL, headers=headers)
    with urllib.request.urlopen(req, timeout=60) as resp:
        raw_bytes = resp.read()
    n_bytes = len(raw_bytes)
    print(f'✓ Downloaded {n_bytes:,} bytes ({n_bytes/1024**2:.2f} MB)')
    
    # Parse CSV directly from bytes
    df_pt = pd.read_csv(io.BytesIO(raw_bytes), low_memory=False)
    df_pt.to_csv(PHISHTANK_RAW, index=False)
    print(f'✓ Saved raw CSV to: {PHISHTANK_RAW}')
    print(f'  Rows: {len(df_pt):,}  Columns: {len(df_pt.columns)}')
    print(f'  Columns: {df_pt.columns.tolist()}')
    print(f'\nFirst row preview:')
    print(df_pt.iloc[0].to_dict())
    
    phishtank_ok = True
    phishtank_n = len(df_pt)
except urllib.error.HTTPError as e:
    print(f'❌ HTTP error {e.code}: {e.reason}')
    if e.code in (403, 429):
        print('  PhishTank rate-limited the request. Wait 15 min and retry.')
    phishtank_ok = False
    phishtank_n = 0
except Exception as e:
    print(f'❌ Error: {type(e).__name__}: {e}')
    phishtank_ok = False
    phishtank_n = 0

### Backup option if PhishTank fails

If the PhishTank download fails (rate limit, network issue), this fallback cell uses **OpenPhish** which is a complementary free phishing URL feed. Only run this cell if Step 1 above failed.

In [ ]:
# Only run this if Step 1 failed.
# OpenPhish provides a free text feed at https://openphish.com/feed.txt

if not phishtank_ok:
    OPENPHISH_URL = 'https://openphish.com/feed.txt'
    print(f'Falling back to OpenPhish: {OPENPHISH_URL}')
    try:
        ctx = ssl.create_default_context()
        req = urllib.request.Request(OPENPHISH_URL, headers={'User-Agent': 'SecureSpeak-NSU-Research'})
        with urllib.request.urlopen(req, timeout=60, context=ctx) as resp:
            urls_text = resp.read().decode('utf-8', errors='ignore')
        urls_list = [u.strip() for u in urls_text.splitlines() if u.strip().startswith('http')]
        df_pt = pd.DataFrame({'url': urls_list, 'verified': 'yes', 'source': 'openphish'})
        df_pt.to_csv(PHISHTANK_RAW, index=False)
        print(f'✓ OpenPhish backup: {len(df_pt):,} URLs saved')
        phishtank_ok = True
        phishtank_n = len(df_pt)
    except Exception as e:
        print(f'❌ OpenPhish also failed: {e}')
        print('   Manual fallback: download from https://openphish.com manually,')
        print(f'   save as: {PHISHTANK_RAW}')
else:
    print('Skipping — PhishTank download was successful.')

---
## Step 2 — Download Benign URLs from Tranco Top Sites

**Why we need benign URLs:** A binary classifier needs both classes. We can't only test it on phishing URLs — we also need legitimate URLs to measure the false-positive rate on the external test set.

**Source:** Tranco (https://tranco-list.eu) — a research-oriented combined ranking of top websites, more stable than the deprecated Alexa Top Sites. Free, no API key needed.

**What we take:** The top 5,000 most-visited domains. These are almost certainly legitimate (Google, Facebook, YouTube, BBC, Wikipedia, etc.).

In [ ]:
# Tranco provides a stable monthly list. We use a recent list ID.
# If this specific list ID expires, we will fall back to the live API endpoint.
TRANCO_URL = 'https://tranco-list.eu/top-1m.csv.zip'
TARGET_BENIGN_N = 5000

print(f'Downloading Tranco Top 1M from: {TRANCO_URL}')
print(f'(we will keep only the top {TARGET_BENIGN_N:,} domains)')
print()

try:
    req = urllib.request.Request(TRANCO_URL, headers={'User-Agent': 'SecureSpeak-NSU-Research'})
    with urllib.request.urlopen(req, timeout=120) as resp:
        zip_bytes = resp.read()
    print(f'✓ Downloaded {len(zip_bytes)/1024**2:.2f} MB (zipped)')
    
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as z:
        # The zip contains one CSV file
        csv_name = z.namelist()[0]
        with z.open(csv_name) as f:
            df_tr = pd.read_csv(f, names=['rank', 'domain'], header=None, nrows=TARGET_BENIGN_N)
    
    # Convert domain → URL (prefix with https://)
    df_tr['url'] = 'https://' + df_tr['domain'].astype(str)
    df_tr = df_tr[['rank', 'domain', 'url']]
    df_tr.to_csv(TRANCO_RAW, index=False)
    print(f'✓ Saved top {len(df_tr):,} Tranco domains to: {TRANCO_RAW}')
    print(f'  Sample: {df_tr["domain"].head(5).tolist()}')
    tranco_ok = True
    tranco_n = len(df_tr)
except Exception as e:
    print(f'❌ Error: {type(e).__name__}: {e}')
    print('   Fallback: We will use a small static list of well-known legitimate sites.')
    tranco_ok = False
    tranco_n = 0

### Backup benign list (only if Tranco failed)

If Tranco fails, this cell builds a small but reliable fallback list of well-known legitimate sites including Bangladeshi domains (so the test set reflects local app-usage patterns).

In [ ]:
if not tranco_ok:
    # Global + Bangladesh-specific legitimate domains
    BACKUP_BENIGN = [
        # Global top sites
        'google.com', 'youtube.com', 'facebook.com', 'wikipedia.org', 'amazon.com',
        'twitter.com', 'instagram.com', 'reddit.com', 'linkedin.com', 'whatsapp.com',
        'github.com', 'stackoverflow.com', 'apple.com', 'microsoft.com', 'netflix.com',
        # Bangladeshi MFS and major sites
        'bkash.com', 'nagad.com.bd', 'rocket.com.bd', 'dbbl.com.bd',
        'prothomalo.com', 'thedailystar.net', 'bdnews24.com', 'banglanews24.com',
        'daraz.com.bd', 'foodpanda.com.bd', 'pathao.com', 'shohoz.com',
        'grameenphone.com', 'robi.com.bd', 'banglalink.net',
        # Educational and government
        'northsouth.edu', 'du.ac.bd', 'buet.ac.bd', 'bangladesh.gov.bd',
        # International banking and finance
        'paypal.com', 'visa.com', 'mastercard.com', 'sc.com',
    ]
    df_tr = pd.DataFrame({
        'rank': range(1, len(BACKUP_BENIGN) + 1),
        'domain': BACKUP_BENIGN,
        'url': ['https://' + d for d in BACKUP_BENIGN],
    })
    df_tr.to_csv(TRANCO_RAW, index=False)
    print(f'✓ Backup benign list saved: {len(df_tr)} URLs (mix of global + Bangladesh)')
    tranco_ok = True
    tranco_n = len(df_tr)
else:
    print('Skipping — Tranco download was successful.')

---
## Step 3 — Clean and Combine into External Test Set

**What we do:**
1. Extract URL columns from both PhishTank and Tranco
2. Strip whitespace, lowercase scheme, remove duplicates
3. Drop URLs that overlap with StealthPhisher training data (prevent leakage)
4. Combine into a single labeled test set: `url`, `label`, `source`
5. Save to `external_url_testset.csv`

**Leakage protection:** This is the critical bit. We do NOT want a PhishTank URL that happens to also be in your StealthPhisher training set. We use URL canonicalization + set intersection to drop overlaps.

In [ ]:
def canonicalize_url(u: str) -> str:
    """Light normalization for deduplication. Not aggressive enough to break domain semantics."""
    if not isinstance(u, str):
        return ''
    u = u.strip().lower()
    # Remove protocol
    for prefix in ('https://', 'http://'):
        if u.startswith(prefix):
            u = u[len(prefix):]
            break
    # Remove leading www.
    if u.startswith('www.'):
        u = u[4:]
    # Strip trailing slash
    return u.rstrip('/')

# ─── Load PhishTank malicious URLs ───────────────────────────────────────────
df_pt = pd.read_csv(PHISHTANK_RAW, low_memory=False)
# PhishTank's column is 'url'; OpenPhish backup uses 'url' too
pt_url_col = next((c for c in df_pt.columns if c.lower() == 'url'), None)
if pt_url_col is None:
    raise ValueError(f'No url column in {PHISHTANK_RAW}. Columns: {df_pt.columns.tolist()}')

phishing_urls = df_pt[pt_url_col].dropna().astype(str).tolist()
print(f'Raw phishing URLs from PhishTank/OpenPhish:  {len(phishing_urls):,}')

# ─── Load Tranco benign URLs ─────────────────────────────────────────────────
df_tr = pd.read_csv(TRANCO_RAW, low_memory=False)
benign_urls = df_tr['url'].dropna().astype(str).tolist()
print(f'Raw benign URLs from Tranco:                 {len(benign_urls):,}')

# ─── Build canonical sets and check overlap with StealthPhisher ─────────────
print('\nLoading StealthPhisher URLs to check for overlap (this may take 30s)...')
STEALTH_CSV = os.path.join(BASE_DATA, 'StealthPhisher2025.csv')
sp_url_col_candidates = ['URL', 'url', 'website']
sp_url_col = None
df_sp_head = pd.read_csv(STEALTH_CSV, nrows=3)
sp_url_col = next((c for c in sp_url_col_candidates if c in df_sp_head.columns), None)
if sp_url_col is None:
    print(f'⚠ No URL column found in StealthPhisher. Columns: {df_sp_head.columns.tolist()}')
    sp_canon_set = set()
else:
    sp_urls = pd.read_csv(STEALTH_CSV, usecols=[sp_url_col], low_memory=False)[sp_url_col].dropna().astype(str)
    sp_canon_set = set(canonicalize_url(u) for u in sp_urls)
    print(f'StealthPhisher canonical URL set built: {len(sp_canon_set):,} unique URLs')

# ─── Deduplicate and remove StealthPhisher overlaps ─────────────────────────
def dedupe_and_filter(urls, name, exclude_set):
    seen = set()
    clean = []
    overlap_count = 0
    for u in urls:
        c = canonicalize_url(u)
        if not c:
            continue
        if c in seen:
            continue
        if c in exclude_set:
            overlap_count += 1
            continue
        seen.add(c)
        clean.append(u.strip())
    print(f'  {name}: {len(urls):,} → {len(clean):,} unique, '
          f'{overlap_count} dropped due to StealthPhisher overlap')
    return clean, overlap_count

print('\nDeduplicating and filtering:')
phishing_clean, pt_overlap = dedupe_and_filter(phishing_urls, 'PhishTank phishing', sp_canon_set)
benign_clean,   tr_overlap = dedupe_and_filter(benign_urls,   'Tranco benign',     sp_canon_set)

# ─── Build combined test set ────────────────────────────────────────────────
external_test = pd.DataFrame({
    'url':    phishing_clean + benign_clean,
    'label':  [1] * len(phishing_clean) + [0] * len(benign_clean),
    'source': ['phishtank'] * len(phishing_clean) + ['tranco'] * len(benign_clean),
})
external_test.to_csv(EXTERNAL_TEST, index=False)

print(f'\n✓ External test set saved: {EXTERNAL_TEST}')
print(f'  Total URLs: {len(external_test):,}')
print(f'  Phishing (label=1): {(external_test.label == 1).sum():,}')
print(f'  Benign   (label=0): {(external_test.label == 0).sum():,}')

---
## Step 4 — Provenance & Quality Report

Reviewers at Q1 venues require **provenance**: when was the data collected, from where, what was the version, how was it cleaned. We save all of that here so we never have to remember it later — the paper's Methodology section will quote from this file directly.

In [ ]:
lines = [
    '═' * 70,
    ' PHISHTANK / TRANCO EXTERNAL TEST SET — ACQUISITION RECORD',
    '═' * 70,
    '',
    f'Acquired at:        {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}',
    f'PhishTank source:   {PHISHTANK_URL}',
    f'Tranco source:      {TRANCO_URL}',
    '',
    'INPUTS',
    f'  PhishTank raw rows:        {phishtank_n:,}',
    f'  Tranco raw rows (top-N):   {tranco_n:,}',
    '',
    'PROCESSING',
    f'  URL canonicalization:      lowercase, strip scheme + www., strip trailing /',
    f'  Dedup within each source:  applied',
    f'  Overlap check vs StealthPhisher 2025:',
    f'    StealthPhisher canonical set size:   {len(sp_canon_set):,}',
    f'    PhishTank rows dropped as overlap:   {pt_overlap}',
    f'    Tranco rows dropped as overlap:      {tr_overlap}',
    '',
    'FINAL EXTERNAL TEST SET',
    f'  Total URLs:                {len(external_test):,}',
    f'  Phishing (label=1):        {(external_test.label == 1).sum():,}',
    f'  Benign   (label=0):        {(external_test.label == 0).sum():,}',
    f'  Saved to:                  {EXTERNAL_TEST}',
    '',
    'NEXT STEP',
    '  Phase 2.2 — Score these URLs with your trained URL classifier and report',
    '  honest external-validation metrics (accuracy, precision, recall, F1).',
    '  Expected accuracy: 75-92% (down from 99.76% on the internal test).',
    '  This drop is the honest cost of generalization and is what reviewers expect.',
]

with open(PROVENANCE_TXT, 'w', encoding='utf-8') as f:
    f.write('\n'.join(lines))

summary = {
    'phase': '2.1',
    'generated_at': datetime.now().isoformat(),
    'phishtank': {
        'source_url': PHISHTANK_URL,
        'raw_rows': int(phishtank_n),
        'after_clean_and_dedup': int((external_test.label == 1).sum()),
        'dropped_overlap_with_stealthphisher': int(pt_overlap),
    },
    'tranco': {
        'source_url': TRANCO_URL,
        'raw_rows': int(tranco_n),
        'after_clean_and_dedup': int((external_test.label == 0).sum()),
        'dropped_overlap_with_stealthphisher': int(tr_overlap),
    },
    'external_test_set': {
        'path': EXTERNAL_TEST,
        'total_rows': int(len(external_test)),
        'phishing_count': int((external_test.label == 1).sum()),
        'benign_count':   int((external_test.label == 0).sum()),
    },
    'verdict': 'ready_for_scoring' if len(external_test) > 100 else 'too_small_retry',
}
with open(SUMMARY_JSON, 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2)

print('\n'.join(lines))
print(f'\n✓ Provenance saved: {PROVENANCE_TXT}')
print(f'✓ Summary saved:    {SUMMARY_JSON}')

---
## Step 5 — Final Sanity Check
Print the head of the external test set so you can eyeball it and confirm it looks right.

In [ ]:
df_final = pd.read_csv(EXTERNAL_TEST)
print('Sample phishing URLs (label=1):')
for u in df_final[df_final.label == 1]['url'].head(5).tolist():
    print(f'  {u[:100]}')

print('\nSample benign URLs (label=0):')
for u in df_final[df_final.label == 0]['url'].head(5).tolist():
    print(f'  {u[:100]}')

print(f'\nTotal: {len(df_final):,} URLs')
print(f'Class balance: phishing={(df_final.label==1).sum():,}  benign={(df_final.label==0).sum():,}')

print('\n' + '═' * 70)
print(' PHASE 2.1 COMPLETE')
print('═' * 70)
print('Next: Phase 2.2 — Score these URLs with your trained URL classifier.')

---
## What This Notebook Did

- Downloaded fresh phishing URLs from PhishTank (live community-verified feed)
- Downloaded benign URLs from Tranco Top Sites (research-grade ranking)
- Removed any URLs that overlap with your StealthPhisher 2025 training data (leakage prevention)
- Built a clean labeled external test set at `model_for_research/phase2_external/external_url_testset.csv`
- Logged full provenance — when, from where, how many

## What This Notebook Did NOT Do
- Did not train any model
- Did not score URLs (Phase 2.2 does that)
- Did not modify your StealthPhisher or any existing dataset

## What to Send Me Back
Send the contents of `phase2_1_summary.json` after running this. From the summary I'll know:
- How many fresh phishing URLs we got (target: >500 for statistical power)
- How many benign URLs (target: >500 for balanced FPR estimation)
- Whether the overlap check found any leakage

## What Comes Next: Phase 2.2
We will load your existing trained URL classifier from your v5 notebook and **score these external URLs without retraining**. The result will be the honest external-validation accuracy that goes in your paper.